# Fresh-start ingest

One-time clean rebuild of the corpus:
1. Wipe `chroma_db/` and `data/processed/*.json` (keep eval data + source PDFs)
2. Re-parse PDFs through Docling
3. Re-embed all chunks into Chroma with stable content-hash IDs

After this completes, the corpus is fully reproducible — same content always
produces the same chunk_ids, so re-parsing is idempotent.

⚠️ **Cell 2 won't actually delete anything until you set `CONFIRM_DELETE = True`.**

In [2]:
"""Wipe chroma_db/ and data/processed/*.json. Eval data is preserved.

Default mode = DRY RUN: shows what will be deleted without doing it.
Set CONFIRM_DELETE = True, re-run the cell to actually delete.
"""
import shutil
from rag_pipeline.config import cfg, log

CONFIRM_DELETE = True   # ← flip to True, then re-run, to actually delete

to_delete = [
    cfg.CHROMA_PERSIST_DIR,
    cfg.DATA_PROCESSED_DIR / "phase0_chunks.json",
    cfg.DATA_PROCESSED_DIR / "phase1_chunks.json",
    cfg.DATA_PROCESSED_DIR / "ragas_rows_mq.json",
    cfg.DATA_PROCESSED_DIR / "ragas_quickstart_cache.json",
]
to_keep = [
    cfg.EVAL_SET_PATH,
    cfg.PROJECT_ROOT / "src" / "rag_pipeline" / "eval" / "data",
    cfg.DATA_RAW_DIR,
    cfg.EVAL_RESULTS_DIR,
]

print("=== WILL DELETE ===")
for p in to_delete:
    if p.exists():
        kind = "DIR " if p.is_dir() else "FILE"
        print(f"  {kind}  {p}")
    else:
        print(f"  (already gone)  {p}")

print("\n=== WILL KEEP ===")
for p in to_keep:
    mark = "✓" if p.exists() else "?"
    print(f"  {mark}  {p}")

if not CONFIRM_DELETE:
    print("\n⚠️  CONFIRM_DELETE=False — nothing deleted. Set True + re-run to wipe.")
else:
    print("\n🔥 Deleting...")
    for p in to_delete:
        if p.is_dir():
            shutil.rmtree(p, ignore_errors=True)
            log.info(f"  rm -rf {p}")
        elif p.is_file():
            p.unlink()
            log.info(f"  rm {p}")
    print("\n✅ Clean slate ready")

=== WILL DELETE ===
  (already gone)  /home/thimu/github_vs/protoRAG/rag-pipeline/chroma_db
  (already gone)  /home/thimu/github_vs/protoRAG/rag-pipeline/data/processed/phase0_chunks.json
  (already gone)  /home/thimu/github_vs/protoRAG/rag-pipeline/data/processed/phase1_chunks.json
  (already gone)  /home/thimu/github_vs/protoRAG/rag-pipeline/data/processed/ragas_rows_mq.json
  (already gone)  /home/thimu/github_vs/protoRAG/rag-pipeline/data/processed/ragas_quickstart_cache.json

=== WILL KEEP ===
  ✓  /home/thimu/github_vs/protoRAG/rag-pipeline/eval/eval_set.json
  ✓  /home/thimu/github_vs/protoRAG/rag-pipeline/src/rag_pipeline/eval/data
  ✓  /home/thimu/github_vs/protoRAG/rag-pipeline/data/raw
  ✓  /home/thimu/github_vs/protoRAG/rag-pipeline/eval/results

🔥 Deleting...

✅ Clean slate ready


In [3]:
"""

Intial Config, LLM, Embedding Model, Data Corpus, Paths ::::::::::::::::::::::::


Verify config + define shared constants for the rest of the notebook."""

from pathlib import Path
from rag_pipeline.config import cfg, log

log.info(f"Provider:    {cfg.MODEL_PROVIDER}")
log.info(f"LLM:         {cfg.OLLAMA_MODEL}")
log.info(f"Embeddings:  {cfg.OLLAMA_EMBEDDING_MODEL}")

# Where your IPC PDFs live (NOT under data/raw/ — they're in your Downloads folder)
PDF_SOURCE_DIR = Path("/home/thimu/Downloads/pdf_splitter/IPC")
assert PDF_SOURCE_DIR.exists(), f"PDFs not found at {PDF_SOURCE_DIR}"

CHUNKS_CACHE = cfg.DATA_PROCESSED_DIR / "phase1_chunks.json"
COLLECTION   = "IPC_Corpus"

log.info(f"PDF source:  {PDF_SOURCE_DIR}")
log.info(f"PDFs found:  {len(list(PDF_SOURCE_DIR.glob('*.pdf')))}")

2026-06-03 23:44:53,771 - INFO    | rag - Provider:    ollama
2026-06-03 23:44:53,772 - INFO    | rag - LLM:         gemma-4-e4b:latest
2026-06-03 23:44:53,772 - INFO    | rag - Embeddings:  embeddinggemma:latest
2026-06-03 23:44:53,773 - INFO    | rag - PDF source:  /home/thimu/Downloads/pdf_splitter/IPC
2026-06-03 23:44:53,773 - INFO    | rag - PDFs found:  74


In [4]:
"""

Parsing:::::::::::::::::::::

Parse every PDF under PDF_SOURCE_DIR with the production dispatcher.

Wall time: ~15-18 min for 63 IPC PDFs on CPU (Docling falls back from CUDA).
The resulting chunks are saved to phase1_chunks.json so this never has to
run again unless the source PDFs change.
"""
from rag_pipeline.parsers import default_dispatcher, save_chunks_cache

dispatcher = default_dispatcher()
chunks = dispatcher.parse_directory(PDF_SOURCE_DIR)

assert chunks, "Parsing produced 0 chunks — check PDF_SOURCE_DIR"

save_chunks_cache(chunks, CHUNKS_CACHE)
log.info(f"Parsed and cached {len(chunks)} chunks")

/home/thimu/github_vs/protoRAG/rag-pipeline/.venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
2026-06-03 23:45:08,318 - INFO    | rag - Found 74 supported files under /home/thimu/Downloads/pdf_splitter/IPC
Parsing:   0%|          | 0/74 [00:00<?, ?file/s]/home/thimu/github_vs/protoRAG/rag-pipeline/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version t

In [5]:
"""
Embedding:::::::::::::::::::


Create an empty Chroma collection and embed every chunk.

Because Cell 2 wiped chroma_db/, the collection starts empty — no need
for the idempotent dedup logic here. Every chunk gets embedded once.

"""

from rag_pipeline.vectorstore import get_vectorstore
from tqdm import tqdm

vs = get_vectorstore(COLLECTION)
assert vs._collection.count() == 0, "Collection not empty — did Cell 2 actually wipe?"

docs = [c.to_langchain_document() for c in chunks]
ids  = [c.chunk_id for c in chunks]

BATCH = 64
for i in tqdm(range(0, len(chunks), BATCH), desc="Embedding"):
    vs.add_documents(documents=docs[i:i + BATCH], ids=ids[i:i + BATCH])

final_count = vs._collection.count()
log.info(f"Vectorstore '{COLLECTION}' has {final_count} vectors")
assert final_count == len(chunks), f"Expected {len(chunks)} vectors, got {final_count}"

2026-06-03 23:49:52,735 - INFO    | rag - initializing vectorstore (provider = chroma, collection=IPC_Corpus)
2026-06-03 23:49:53,142 - INFO    | rag - Instantiating embedding model for provider: ollama
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Embedding: 100%|██████████| 10/10 [08:09<00:00, 48.99s/it]
2026-06-03 23:58:08,680 - INFO    | rag - Vectorstore 'IPC_Corpus' has 613 vectors


In [6]:
"""
Retriever:::::::::::::::::::::::::::::::::

hybrid_reranked = dense + BM25 + BGE cross-encoder rerank."""

from rag_pipeline.retrievers import (
    BM25Retriever, DenseRetriever, EnsembleRetriever,
    Reranker, RerankedRetriever,
)

dense    = DenseRetriever(collection_name=COLLECTION)
bm25     = BM25Retriever(chunks)
ensemble = EnsembleRetriever([dense, bm25], fetch_k=20)
reranker = Reranker()                                     # downloads BGE on first run
hybrid_r = RerankedRetriever(ensemble, reranker, fetch_k=20, min_score=0.5)

log.info("Retriever stack ready")

2026-06-03 23:58:59,795 - INFO    | rag - Loading reranker: BAAI/bge-reranker-base
2026-06-03 23:59:07,281 - INFO    | rag - Retriever stack ready


In [7]:
"""
Querying::::::::::::::::::::::::::::

Single-query smoke test through the full RAG pipeline."""

from rag_pipeline.providers import get_llm
from rag_pipeline.generation import answer, pretty_print

llm = get_llm()

response = answer(
    query="What is the punishment for cruelty by a husband against his wife?",
    retriever=hybrid_r,
    llm=llm,
    top_k=3,
)
pretty_print(response, score_label="rerank")

2026-06-03 23:59:27,944 - INFO    | rag - Instantiating LLM for provider: ollama
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
2026-06-03 23:59:55,652 - INFO    | rag - PromptManager: loading from /home/thimu/github_vs/protoRAG/rag-pipeline/src/rag_pipeline/prompts/templates


What is the punishment for cruelty by a husband against his wife?

A person who is the husband or relative of the husband of a woman and subjects her to cruelty shall be punished with imprisonment for a term which may extend to three years and shall also be liable to fine [2].

(Note: For the purposes of section 85, 'cruelty' is defined as: (a) any wilful conduct which is of such a nature as is likely to drive the woman to commit suicide or to cause grave injury or danger to life, limb or health (whether mental or physical) of the woman; or (b) harassment of the woman where such harassment is with a view to coercing her or any person related to her to meet any unlawful demand for any property or valuable security or is on account of failure by her or any person related to her to meet such demand [2]).

Sources:
   [1] /home/thimu/Downloads/pdf_splitter/IPC/IPC_split_10.pdf  |  p.3 | § OF OFFENCES AGAINST WOMAN AND CHILD   (rerank=0.849)
   [2] /home/thimu/Downloads/pdf_splitter/IPC/IPC

In [8]:
"""

Retrieval Metrics::::::::::::::::::::::::::::

Score the fresh retriever against the eval set. ~30-60 sec."""

from rag_pipeline.eval import load_eval_set, evaluate_retriever

examples = load_eval_set(cfg.EVAL_SET_PATH)
results  = evaluate_retriever(hybrid_r, examples, top_k=5)

print(f"\n=== Retrieval metrics ({results['n_positive']} positives) ===\nOverall:")
for k, v in results["overall"].items():
    print(f"  {k:8s}: {v:.3f}")

if results["by_difficulty"]:
    print("\nBy difficulty:")
    for diff, m in results["by_difficulty"].items():
        row = "  ".join(f"{k}={v:.3f}" for k, v in m.items())
        print(f"  {diff:6s}: {row}")

2026-06-04 00:04:19,561 - INFO    | rag - Loaded 96 eval examples ← eval_set.json



=== Retrieval metrics (90 positives) ===
Overall:
  hit     : 0.411
  recall  : 0.383
  mrr     : 0.341
  snippet : 0.356

By difficulty:
  easy  : hit=0.950  recall=0.950  mrr=0.833  snippet=0.950
  medium: hit=0.433  recall=0.433  mrr=0.339  snippet=0.433
  hard  : hit=0.125  recall=0.062  mrr=0.096  snippet=0.000


In [9]:
"""
Refusal rate check:::::::::::::::::::::::::


Every negative should produce a refusal. Phase 4 baseline = 100%."""
from rag_pipeline.eval import load_negatives, is_refusal

negatives = load_negatives()
refusals = 0
for neg in negatives:
    resp = answer(neg.question, hybrid_r, llm, top_k=3)
    refused = is_refusal(resp["answer"])
    refusals += int(refused)
    print(f"  {'good' if refused else 'bad'}  {neg.question[:80]}")

print(f"\nRefusal rate: {refusals}/{len(negatives)} ({100 * refusals / len(negatives):.0f}%)")

2026-06-04 00:31:01,956 - INFO    | rag - Loaded 7 negative examples ← ipc_negatives.yaml
2026-06-04 00:31:17,706 - INFO    | rag - Loaded 6 refusal markers ← refusal_markers.yaml


  good  How many days of parental leave are employees entitled to in India?
  good  What is the current GST rate on luxury items?
  good  How many members must a startup team have?
  good  What is the procedure for filing an FIR under the CrPC?
  good  What does the IT Act say about cybercrime penalties?
  good  What does Article 21 of the Constitution guarantee?
  good  Under the Indian Evidence Act, what is the rule regarding hearsay?

Refusal rate: 7/7 (100%)
